<a href="https://colab.research.google.com/github/sarita380/5BTelemedicineAppSol/blob/master/Homework6_PCA_Spark_Scala.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless
!java -version


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  openjdk-17-jre-headless
Suggested packages:
  openjdk-17-demo openjdk-17-source libnss-mdns fonts-dejavu-extra
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic
The following NEW packages will be installed:
  openjdk-17-jdk-headless openjdk-17-jre-headless
0 upgraded, 2 newly installed, 0 to remove and 1 not upgraded.
Need to get 120 MB of archives.
After this operation, 272 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 openjdk-17-jre-headless amd64 17.0.20.1+1-1~24.04 [48.1 MB]
Get:2 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 openj

In [2]:

!wget -q https://archive.apache.org/dist/spark/spark-3.5.7/spark-3.5.7-bin-hadoop3.tgz
!tar -xzf spark-3.5.7-bin-hadoop3.tgz
!mv spark-3.5.7-bin-hadoop3 /content/spark
!/content/spark/bin/spark-shell --version


[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 3.5.7
      /_/
                        
Using Scala version 2.12.18, OpenJDK 64-Bit Server VM, 21.0.12.1
Branch HEAD
Compiled by user runner on 2025-09-17T20:37:30Z
Revision ed00d046951a7ecda6429accd3b9c5b2d

In [3]:

import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark"

os.environ["PATH"] = (
    os.environ["JAVA_HOME"] + "/bin:" +
    os.environ["SPARK_HOME"] + "/bin:" +
    os.environ["PATH"]
)

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("SPARK_HOME:", os.environ["SPARK_HOME"])
print("Environment configured!")


JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64
SPARK_HOME: /content/spark
Environment configured!


Upload exPCA.mat and jmatio.jar

In [6]:

from google.colab import files

uploaded = files.upload()



Saving jmatio.jar to jmatio (2).jar
Saving exPCA.mat to exPCA.mat


In [7]:

import os

print("MATLAB dataset:",
      os.path.exists("/content/exPCA.mat"))

print("JMatIO library:",
      os.path.exists("/content/jmatio.jar"))


MATLAB dataset: True
JMatIO library: True


Matlab using Scale

In [8]:

scala_code = """
import com.jmatio.io.MatFileReader
import com.jmatio.types.MLDouble

val reader = new MatFileReader("/content/exPCA.mat")
val content = reader.getContent()

val data = content.get("X")
  .asInstanceOf[MLDouble].getArray()

println("Number of rows: " + data.length)
println("Number of columns: " + data(0).length)

println("First 5 observations:")
data.take(5).foreach(row => println(row.mkString(", ")))

System.exit(0)
"""

with open("/content/check_data.scala", "w") as f:
    f.write(scala_code)

import subprocess

result = subprocess.run(
    ["/content/spark/bin/spark-shell",
     "--jars", "/content/jmatio.jar",
     "-i", "/content/check_data.scala"],
    input=":quit\n",
    text=True,
    capture_output=True,
    timeout=180
)

print(result.stdout[-5000:])
print(result.stderr[-1500:])
print("Exit code:", result.returncode)


Spark context Web UI available at http://49fffdef7fb9:4040
Spark context available as 'sc' (master = local[*], app id = local-1793159127147).
Spark session available as 'spark'.
java.lang.NoClassDefFoundError: sun/misc/Cleaner
  at com.jmatio.io.MatFileReader$1.run(Unknown Source)
  at java.base/java.security.AccessController.doPrivileged(AccessController.java:318)
  at com.jmatio.io.MatFileReader.clean(Unknown Source)
  at com.jmatio.io.MatFileReader.read(Unknown Source)
  at com.jmatio.io.MatFileReader.<init>(Unknown Source)
  at com.jmatio.io.MatFileReader.<init>(Unknown Source)
  ... 60 elided
Caused by: java.lang.ClassNotFoundException: sun.misc.Cleaner
  at java.base/java.net.URLClassLoader.findClass(URLClassLoader.java:445)
  at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:592)
  at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:525)
  ... 66 more
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ ._

In [9]:

import os
import subprocess

# Download a newer JMatIO version from Maven Central
url = (
    "https://repo.maven.apache.org/maven2/"
    "net/sourceforge/jmatio/jmatio/1.0/"
    "jmatio-1.0.jar"
)

result = subprocess.run(
    ["wget", "-O", "/content/jmatio-new.jar", url],
    capture_output=True,
    text=True
)

print("Download exit code:", result.returncode)

if result.returncode == 0:
    print("Library size:",
          os.path.getsize("/content/jmatio-new.jar"))
else:
    print(result.stderr[-2000:])



Download exit code: 0
Library size: 66536


In [10]:

import subprocess

result = subprocess.run(
    [
        "/content/spark/bin/spark-shell",
        "--jars", "/content/jmatio-new.jar",
        "-i", "/content/check_data.scala"
    ],
    input=":quit\n",
    text=True,
    capture_output=True,
    timeout=180
)

print(result.stdout[-5000:])
print(result.stderr[-1500:])
print("Exit code:", result.returncode)


Spark context Web UI available at http://49fffdef7fb9:4040
Spark context available as 'sc' (master = local[*], app id = local-1793159399947).
Spark session available as 'spark'.
java.lang.NoClassDefFoundError: sun/misc/Cleaner
  at com.jmatio.io.MatFileReader$1.run(MatFileReader.java:375)
  at java.base/java.security.AccessController.doPrivileged(AccessController.java:318)
  at com.jmatio.io.MatFileReader.clean(MatFileReader.java:366)
  at com.jmatio.io.MatFileReader.read(MatFileReader.java:321)
  at com.jmatio.io.MatFileReader.<init>(MatFileReader.java:154)
  at com.jmatio.io.MatFileReader.<init>(MatFileReader.java:102)
  ... 60 elided
Caused by: java.lang.ClassNotFoundException: sun.misc.Cleaner
  at java.base/java.net.URLClassLoader.findClass(URLClassLoader.java:445)
  at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:592)
  at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:525)
  ... 66 more
Welcome to
      ____              __
     / __/__  ___ _____/ /__


In [11]:

import subprocess

result = subprocess.run(
    [
        "/content/spark/bin/spark-shell",
        "--jars", "/content/jmatio-new.jar",
        "-i", "/content/check_data.scala"
    ],
    input=":quit\n",
    text=True,
    capture_output=True,
    timeout=180
)

print("=== SCALA OUTPUT ===")
print(result.stdout[-6000:])

print("=== ERRORS ===")
print(result.stderr[-2000:])

print("Exit code:", result.returncode)


=== SCALA OUTPUT ===
Spark context Web UI available at http://49fffdef7fb9:4040
Spark context available as 'sc' (master = local[*], app id = local-1793159519243).
Spark session available as 'spark'.
java.lang.NoClassDefFoundError: sun/misc/Cleaner
  at com.jmatio.io.MatFileReader$1.run(MatFileReader.java:375)
  at java.base/java.security.AccessController.doPrivileged(AccessController.java:318)
  at com.jmatio.io.MatFileReader.clean(MatFileReader.java:366)
  at com.jmatio.io.MatFileReader.read(MatFileReader.java:321)
  at com.jmatio.io.MatFileReader.<init>(MatFileReader.java:154)
  at com.jmatio.io.MatFileReader.<init>(MatFileReader.java:102)
  ... 60 elided
Caused by: java.lang.ClassNotFoundException: sun.misc.Cleaner
  at java.base/java.net.URLClassLoader.findClass(URLClassLoader.java:445)
  at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:592)
  at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:525)
  ... 66 more
Welcome to
      ____              __
     / _

Convert MATLAB to CSV

In [12]:

import scipy.io
import numpy as np
import pandas as pd

# Load MATLAB file
mat = scipy.io.loadmat("/content/exPCA.mat")

# Display variables
print("MATLAB variables:", mat.keys())

# Extract dataset
X = np.asarray(mat["X"], dtype=float)

print("Dataset shape:", X.shape)

# Convert to CSV without headers
pd.DataFrame(X).to_csv(
    "/content/exPCA.csv",
    index=False,
    header=False
)

print("CSV file created successfully!")
print("First 5 observations:")
print(X[:5])


MATLAB variables: dict_keys(['__header__', '__version__', '__globals__', 'X'])
Dataset shape: (50, 2)
CSV file created successfully!
First 5 observations:
[[3.38156267 3.38911268]
 [4.52787538 5.8541781 ]
 [2.65568187 4.41199472]
 [2.76523467 3.71541365]
 [2.84656011 4.17550645]]


Verify the CSV

In [13]:

import pandas as pd

df = pd.read_csv(
    "/content/exPCA.csv",
    header=None,
    names=["Feature_1", "Feature_2"]
)

print("Number of observations:", len(df))
print("Number of features:", len(df.columns))

display(df.head())


Number of observations: 50
Number of features: 2


,Feature_1,Feature_2
0,3.381563,3.389113
1,4.527875,5.854178
2,2.655682,4.411995
3,2.765235,3.715414
4,2.846560,4.175506


**Normalize the data using Scala**
Create and run the Scala normalization program

In [14]:

import subprocess

scala_code = """
import org.apache.spark.mllib.linalg.Vectors

// Load CSV data
val rawData = sc.textFile("/content/exPCA.csv")
  .map(line => line.split(",").map(_.toDouble))
  .cache()

// Calculate feature means
val n = rawData.count()

val mean1 = rawData.map(_(0)).sum() / n
val mean2 = rawData.map(_(1)).sum() / n

println("===== FEATURE MEANS =====")
println(s"Feature 1 Mean: $mean1")
println(s"Feature 2 Mean: $mean2")

// Normalize by subtracting means
val normalized = rawData.map { row =>
  Vectors.dense(
    row(0) - mean1,
    row(1) - mean2
  )
}.cache()

println("===== FIRST 5 NORMALIZED ROWS =====")
normalized.take(5).foreach(println)

// Verify normalized means
val check1 = normalized.map(_(0)).sum() / n
val check2 = normalized.map(_(1)).sum() / n

println("===== NORMALIZED MEANS =====")
println(s"Feature 1: $check1")
println(s"Feature 2: $check2")
"""

with open("/content/normalize.scala", "w") as f:
    f.write(scala_code)

result = subprocess.run(
    ["/content/spark/bin/spark-shell",
     "-i", "/content/normalize.scala"],
    input=":quit\n",
    text=True,
    capture_output=True,
    timeout=180
)

print(result.stdout[-7000:])
print(result.stderr[-1000:])


Spark context Web UI available at http://49fffdef7fb9:4040
Spark context available as 'sc' (master = local[*], app id = local-1793159928650).
Spark session available as 'spark'.
===== FEATURE MEANS =====
Feature 1 Mean: 3.9892652815843146
Feature 2 Mean: 5.002805850257663
===== FIRST 5 NORMALIZED ROWS =====
[-0.6077026149487548,-1.613693165365592]
[0.5386100988188929,0.8513722509117505]
[-1.3335834128448516,-0.5908111327728776]
[-1.2240306073592393,-1.287392200514371]
[-1.1427051753632202,-0.8272994007432777]
===== NORMALIZED MEANS =====
Feature 1: -1.0480505352461477E-15
Feature 2: 2.575717417130363E-16
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 3.5.7
      /_/
         
Using Scala version 2.12.18 (OpenJDK 64-Bit Server VM, Java 17.0.20.1)
Type in expressions to have them evaluated.
Type :help for more information.

scala> :quit

Setting default log level to "WARN".
To adjust logging level use 

**Calculate the top principal component using Spark PCA**

In [15]:

import subprocess

scala_code = """
import org.apache.spark.mllib.linalg.Vectors
import org.apache.spark.mllib.linalg.distributed.RowMatrix

// Load the dataset
val rawData = sc.textFile("/content/exPCA.csv")
  .map(line => line.split(",").map(_.toDouble))
  .cache()

// Calculate means
val n = rawData.count()
val mean1 = rawData.map(_(0)).sum() / n
val mean2 = rawData.map(_(1)).sum() / n

// Normalize the data
val normalized = rawData.map { row =>
  Vectors.dense(
    row(0) - mean1,
    row(1) - mean2
  )
}

// Create Spark RowMatrix
val matrix = new RowMatrix(normalized)

// Compute top principal component
val pc = matrix.computePrincipalComponents(1)

println()
println("================================")
println("   TOP PRINCIPAL COMPONENT")
println("================================")
println(pc)
println("================================")
"""

with open("/content/pca.scala", "w") as f:
    f.write(scala_code)

result = subprocess.run(
    [
        "/content/spark/bin/spark-shell",
        "-i", "/content/pca.scala"
    ],
    input=":quit\n",
    text=True,
    capture_output=True,
    timeout=180
)

print(result.stdout[-6000:])
print(result.stderr[-1500:])


Spark context Web UI available at http://49fffdef7fb9:4040
Spark context available as 'sc' (master = local[*], app id = local-1793160064623).
Spark session available as 'spark'.

   TOP PRINCIPAL COMPONENT
-0.76908153413682    
-0.6391506816469457  
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 3.5.7
      /_/
         
Using Scala version 2.12.18 (OpenJDK 64-Bit Server VM, Java 17.0.20.1)
Type in expressions to have them evaluated.
Type :help for more information.

scala> :quit

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/28 04:01:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable

[Stage 0:=============================>                             (1 + 1) / 2]

                                                                       

**Project all 50 observations onto PC1** | Calculate the 50 projected values

In [16]:

import subprocess

scala_code = """
import org.apache.spark.mllib.linalg.Vectors
import org.apache.spark.mllib.linalg.distributed.RowMatrix

// Load CSV data
val rawData = sc.textFile("/content/exPCA.csv")
  .map(line => line.split(",").map(_.toDouble))
  .cache()

// Calculate feature means
val n = rawData.count()
val mean1 = rawData.map(_(0)).sum() / n
val mean2 = rawData.map(_(1)).sum() / n

// Normalize data
val normalized = rawData.map { row =>
  Vectors.dense(
    row(0) - mean1,
    row(1) - mean2
  )
}

// Create RowMatrix
val matrix = new RowMatrix(normalized)

// Calculate top principal component
val pc = matrix.computePrincipalComponents(1)

// Project data onto PC1
val projected = matrix.multiply(pc)

// Collect all 50 projected values
val values = projected.rows.collect()

println()
println("================================")
println(" PROJECTED DATA - 50 VALUES")
println("================================")

values.zipWithIndex.foreach {
  case (v, i) =>
    println(f"${i + 1}%2d: ${v(0)}%.8f")
}

println("================================")
println("Total projected values: " + values.length)
"""

with open("/content/projection.scala", "w") as f:
    f.write(scala_code)

result = subprocess.run(
    [
        "/content/spark/bin/spark-shell",
        "-i", "/content/projection.scala"
    ],
    input=":quit\n",
    text=True,
    capture_output=True,
    timeout=180
)

print(result.stdout[-10000:])
print(result.stderr[-1000:])


Spark context Web UI available at http://49fffdef7fb9:4040
Spark context available as 'sc' (master = local[*], app id = local-1793160410036).
Spark session available as 'spark'.

 PROJECTED DATA - 50 VALUES
 1: 1.49876595
 2: -0.95839024
 3: 1.40325172
 4: 1.76421694
 5: 1.40760243
 6: -0.87367998
 7: 1.27050164
 8: -2.55067120
 9: -0.01469839
10: -0.83694188
11: -0.70212917
12: -0.58711016
13: -0.12493311
14: -0.74690506
15: 1.67629396
16: -2.10275704
17: -0.95949530
18: 1.11633715
19: -2.37070273
20: -0.69001651
21: -2.39397485
22: 0.44284714
23: -1.98340505
24: -0.01058959
25: 1.83205377
26: 0.62719172
27: -1.33171608
28: -1.45467270
29: 1.01919098
30: 0.01489202
31: -0.07212622
32: -1.78539513
33: 1.41318051
34: -0.82644523
35: 0.75167377
36: -1.40551081
37: 1.82309802
38: -1.59458841
39: 2.80783613
40: -0.32551527
41: -0.98578762
42: 0.98465469
43: 1.38952836
44: -1.03742062
45: 1.87686597
46: -0.24535117
47: 3.51800218
48: 1.54860441
49: 0.34412682
50: -1.55978675
Total projected

In [17]:
!/content/spark/bin/spark-shell -i /content/Homework6_PCA.scala

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/28 04:21:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Spark context Web UI available at http://49fffdef7fb9:4040
Spark context available as 'sc' (master = local[*], app id = local-1793161282523).
Spark session available as 'spark'.
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 3.5.7
      /_/
         
Using Scala version 2.12.18 (OpenJDK 64-Bit Server VM, Java 17.0.20.1)
Type in expressions to have them evaluated.
Type :help for more information.

scala> 

In [19]:

from google.colab import files

uploaded = files.upload()


Saving exPCA.csv to exPCA (1).csv
Saving Homework6_PCA.scala to Homework6_PCA.scala


In [20]:

import os

print("Scala source file:",
      os.path.exists("/content/Homework6_PCA.scala"))

print("CSV dataset:",
      os.path.exists("/content/exPCA.csv"))


Scala source file: True
CSV dataset: True


In [21]:

import subprocess

result = subprocess.run(
    [
        "/content/spark/bin/spark-shell",
        "-i", "/content/Homework6_PCA.scala"
    ],
    input=":quit\n",
    text=True,
    capture_output=True,
    timeout=180
)

print(result.stdout[-12000:])
print(result.stderr[-2000:])
print("Exit code:", result.returncode)


Spark context Web UI available at http://49fffdef7fb9:4040
Spark context available as 'sc' (master = local[*], app id = local-1793161554239).
Spark session available as 'spark'.
Original means: 3.9892652815843146, 5.002805850257663
Centered means: -1.0480505352461477E-15, 2.575717417130363E-16
===== TOP PRINCIPAL COMPONENT =====
-0.76908153413682    
-0.6391506816469457  
===== PROJECTED DATA - 50 VALUES =====
 1: 1.49876595
 2: -0.95839024
 3: 1.40325172
 4: 1.76421694
 5: 1.40760243
 6: -0.87367998
 7: 1.27050164
 8: -2.55067120
 9: -0.01469839
10: -0.83694188
11: -0.70212917
12: -0.58711016
13: -0.12493311
14: -0.74690506
15: 1.67629396
16: -2.10275704
17: -0.95949530
18: 1.11633715
19: -2.37070273
20: -0.69001651
21: -2.39397485
22: 0.44284714
23: -1.98340505
24: -0.01058959
25: 1.83205377
26: 0.62719172
27: -1.33171608
28: -1.45467270
29: 1.01919098
30: 0.01489202
31: -0.07212622
32: -1.78539513
33: 1.41318051
34: -0.82644523
35: 0.75167377
36: -1.40551081
37: 1.82309802
38: -1.59

**Create a Scala Program for HW 6 **

In [1]:

scala_code = r'''
import java.nio.{ByteBuffer, ByteOrder}
import java.nio.file.{Files, Paths}
import java.io.{ByteArrayInputStream, ByteArrayOutputStream}
import java.util.zip.InflaterInputStream

import org.apache.spark.mllib.linalg.Vectors
import org.apache.spark.mllib.linalg.distributed.RowMatrix

// MATLAB Level 5 binary element reader
def element(b: ByteBuffer): (Int, Array[Byte]) = {
  val tag = b.getInt()
  val smallSize = tag >>> 16
  val kind = tag & 0xffff

  if (smallSize != 0) {
    val four = new Array[Byte](4)
    b.get(four)
    (kind, four.take(smallSize))
  } else {
    val size = b.getInt()
    val bytes = new Array[Byte](size)
    b.get(bytes)
    val padding = (8 - size % 8) % 8
    b.position(b.position() + padding)
    (tag, bytes)
  }
}

def buffer(bytes: Array[Byte]): ByteBuffer =
  ByteBuffer.wrap(bytes).order(ByteOrder.LITTLE_ENDIAN)

def decompress(bytes: Array[Byte]): Array[Byte] = {
  val input = new InflaterInputStream(
    new ByteArrayInputStream(bytes))
  val output = new ByteArrayOutputStream()
  val chunk = new Array[Byte](4096)
  var count = input.read(chunk)
  while (count != -1) {
    output.write(chunk, 0, count)
    count = input.read(chunk)
  }
  input.close()
  output.toByteArray
}

// Read the X variable from exPCA.mat
val fileBytes = Files.readAllBytes(
  Paths.get("/content/exPCA.mat"))

val fileBuffer = buffer(fileBytes)
fileBuffer.position(128)

var matrixBytes: Array[Byte] = Array.emptyByteArray

while (fileBuffer.remaining() >= 8 &&
       matrixBytes.isEmpty) {
  val (kind, payload) = element(fileBuffer)

  if (kind == 15) {
    val uncompressed = buffer(decompress(payload))
    val (innerKind, innerPayload) = element(uncompressed)
    if (innerKind == 14) matrixBytes = innerPayload
  } else if (kind == 14) {
    matrixBytes = payload
  }
}

require(matrixBytes.nonEmpty, "MATLAB matrix not found")

val mat = buffer(matrixBytes)
val flags = element(mat)
val dimensions = element(mat)
val name = element(mat)
val realData = element(mat)

val dims = buffer(dimensions._2)
val numRows = dims.getInt()
val numCols = dims.getInt()

val variableName = new String(name._2, "US-ASCII")
require(variableName == "X", "Expected variable X")
require(realData._1 == 9, "Expected double data")

val values = buffer(realData._2)
val X = Array.ofDim[Double](numRows, numCols)

// MATLAB stores matrix values column by column
for (j <- 0 until numCols; i <- 0 until numRows) {
  X(i)(j) = values.getDouble()
}

println("===== MATLAB DATA LOADED IN SCALA =====")
println(s"Rows: $numRows")
println(s"Columns: $numCols")
X.take(5).foreach(r => println(r.mkString(", ")))

require(numRows == 50 && numCols == 2)

// Normalize the data
val mean1 = X.map(_(0)).sum / numRows
val mean2 = X.map(_(1)).sum / numRows

val centered = X.map(r =>
  Vectors.dense(r(0) - mean1, r(1) - mean2))

println("===== ORIGINAL MEANS =====")
println(s"Feature 1: $mean1")
println(s"Feature 2: $mean2")

// PCA with Spark
val rows = sc.parallelize(centered.toSeq)
val matrix = new RowMatrix(rows)
val pc = matrix.computePrincipalComponents(1)

println("===== TOP PRINCIPAL COMPONENT =====")
println(pc)

// Project observations
val projected = matrix.multiply(pc).rows.collect()

println("===== PROJECTED DATA - 50 VALUES =====")
projected.zipWithIndex.foreach {
  case (v, i) =>
    println(f"${i + 1}%2d: ${v(0)}%.8f")
}

println("Total projected values: " + projected.length)
'''

with open("/content/Homework6_PCA_Direct.scala", "w") as f:
    f.write(scala_code)

print("Scala source file created successfully!")


Scala source file created successfully!


**Scala in colab**

In [2]:

import subprocess

result = subprocess.run(
    [
        "/content/spark/bin/spark-shell",
        "-i", "/content/Homework6_PCA_Direct.scala"
    ],
    input=":quit\n",
    text=True,
    capture_output=True,
    timeout=180
)

print(result.stdout[-12000:])
print(result.stderr[-2000:])
print("Exit code:", result.returncode)


FileNotFoundError: [Errno 2] No such file or directory: '/content/spark/bin/spark-shell'

# **Install Spark **

In [3]:

import os

print("Spark installed:",
      os.path.exists("/content/spark/bin/spark-shell"))

print("Scala source file:",
      os.path.exists("/content/Homework6_PCA_Direct.scala"))

print("MATLAB dataset:",
      os.path.exists("/content/exPCA.mat"))


Spark installed: False
Scala source file: True
MATLAB dataset: False


**Reinstall Spark 3.5.7**

In [5]:

!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless

!wget -q https://archive.apache.org/dist/spark/spark-3.5.7/spark-3.5.7-bin-hadoop3.tgz

!tar -xzf spark-3.5.7-bin-hadoop3.tgz

!mv spark-3.5.7-bin-hadoop3 /content/spark

!/content/spark/bin/spark-shell --version


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-17-jdk-headless is already the newest version (17.0.20.1+1-1~24.04).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.
^C

gzip: stdin: unexpected end of file
tar: Unexpected EOF in archive
tar: Unexpected EOF in archive
tar: Error is not recoverable: exiting now
/bin/bash: line 1: /content/spark/bin/spark-shell: No such file or directory


Check java Spark

In [6]:

import os
import subprocess

print("===== JAVA CHECK =====")
java = subprocess.run(
    ["java", "-version"],
    capture_output=True,
    text=True
)
print(java.stderr)
print("Java exit code:", java.returncode)

print("\n===== SPARK CHECK =====")
spark_path = "/content/spark/bin/spark-shell"
print("Spark exists:", os.path.exists(spark_path))

print("\n===== HOMEWORK FILES =====")
print("Scala file:", os.path.exists(
    "/content/Homework6_PCA_Direct.scala"))
print("MATLAB file:", os.path.exists(
    "/content/exPCA.mat"))


===== JAVA CHECK =====
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)

Java exit code: 0

===== SPARK CHECK =====
Spark exists: False

===== HOMEWORK FILES =====
Scala file: True
MATLAB file: False


Install Apache Spark

In [7]:

import os
import subprocess

# Download Spark only if it is missing
if not os.path.exists("/content/spark/bin/spark-shell"):
    subprocess.run(
        [
            "wget", "-q",
            "https://archive.apache.org/dist/spark/"
            "spark-3.5.7/spark-3.5.7-bin-hadoop3.tgz",
            "-O", "/content/spark.tgz"
        ],
        check=True
    )

    subprocess.run(
        ["tar", "-xzf", "/content/spark.tgz",
         "-C", "/content"],
        check=True
    )

    os.rename(
        "/content/spark-3.5.7-bin-hadoop3",
        "/content/spark"
    )

os.environ["SPARK_HOME"] = "/content/spark"
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"

print("Spark installed:",
      os.path.exists("/content/spark/bin/spark-shell"))


KeyboardInterrupt: 

Diagnostic

In [8]:

import os

path = "/content/spark.tgz"

if os.path.exists(path):
    size = os.path.getsize(path) / (1024 * 1024)
    print(f"Downloaded file size: {size:.2f} MB")
else:
    print("Spark download file not found.")

print("Spark installed:",
      os.path.exists("/content/spark/bin/spark-shell"))


Downloaded file size: 298.19 MB
Spark installed: False


Download Spark Active Checking

In [9]:

import subprocess
import os

archive = "/content/spark.tgz"

result = subprocess.run(
    ["tar", "-tzf", archive],
    capture_output=True,
    text=True,
    timeout=120
)

print("Archive valid:", result.returncode == 0)

if result.returncode != 0:
    print("Error:", result.stderr[-1000:])
else:
    print("Spark archive is ready for extraction!")


Archive valid: False
Error: 
gzip: stdin: unexpected end of file
tar: Unexpected EOF in archive
tar: Error is not recoverable: exiting now



Resume Spark install

In [10]:

import subprocess

url = (
    "https://archive.apache.org/dist/spark/"
    "spark-3.5.7/spark-3.5.7-bin-hadoop3.tgz"
)

result = subprocess.run(
    [
        "wget",
        "-c",
        "--progress=bar:force",
        "-O", "/content/spark.tgz",
        url
    ],
    text=True
)

print("Download exit code:", result.returncode)


Download exit code: 0


Verify the downloaded file

In [11]:

import os
import subprocess

result = subprocess.run(
    ["tar", "-tzf", "/content/spark.tgz"],
    capture_output=True,
    text=True,
    timeout=120
)

print("Archive valid:", result.returncode == 0)
print("File size:",
      round(os.path.getsize("/content/spark.tgz") / 1024**2, 2),
      "MB")

if result.returncode != 0:
    print(result.stderr[-1000:])


Archive valid: True
File size: 382.34 MB


Extract Apache Spark

In [12]:

import os
import subprocess

source = "/content/spark-3.5.7-bin-hadoop3"
destination = "/content/spark"

if not os.path.exists(destination + "/bin/spark-shell"):
    subprocess.run(
        ["tar", "-xzf", "/content/spark.tgz",
         "-C", "/content"],
        check=True
    )

    if os.path.exists(source) and not os.path.exists(destination):
        os.rename(source, destination)

print("Spark installed:",
      os.path.exists("/content/spark/bin/spark-shell"))



Spark installed: False


Locate the extracted Spark folder

In [13]:

import os

print("Spark-related folders:")
for name in os.listdir("/content"):
    if "spark" in name.lower():
        path = os.path.join("/content", name)
        print(name, "- Directory:", os.path.isdir(path))

print("\nScala shell locations:")
for path in [
    "/content/spark/bin/spark-shell",
    "/content/spark-3.5.7-bin-hadoop3/bin/spark-shell"
]:
    print(path, os.path.exists(path))


Spark-related folders:
spark.tgz - Directory: False
spark-3.5.7-bin-hadoop3.tgz - Directory: False
spark - Directory: True
spark-3.5.7-bin-hadoop3.tgz.1 - Directory: False
spark-3.5.7-bin-hadoop3 - Directory: True

Scala shell locations:
/content/spark/bin/spark-shell False
/content/spark-3.5.7-bin-hadoop3/bin/spark-shell True


Fix the Spark folder

In [14]:

import os

source = "/content/spark-3.5.7-bin-hadoop3"
destination = "/content/spark"

if os.path.exists(source + "/bin/spark-shell"):
    os.environ["SPARK_HOME"] = source
    print("Using extracted Spark folder directly.")
elif os.path.exists(destination + "/bin/spark-shell"):
    os.environ["SPARK_HOME"] = destination
    print("Using existing Spark folder.")
else:
    print("Spark shell not found. Check Step 1 output.")

print("SPARK_HOME:", os.environ.get("SPARK_HOME"))
print("Spark ready:", os.path.exists(
    os.path.join(os.environ.get("SPARK_HOME", ""), "bin", "spark-shell")
))


Using extracted Spark folder directly.
SPARK_HOME: /content/spark-3.5.7-bin-hadoop3
Spark ready: True


Verify Spark and Scala

In [15]:

import os
import subprocess

spark_shell = os.path.join(
    os.environ["SPARK_HOME"], "bin", "spark-shell"
)

result = subprocess.run(
    [spark_shell, "--version"],
    capture_output=True,
    text=True,
    timeout=60
)

print(result.stdout[-1500:])
print(result.stderr[-1500:])
print("Exit code:", result.returncode)


[0.004s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.005s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate

Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 3.5.7
      /_/
                        
Using Scala version 2.12.18, OpenJDK 64-Bit Server VM, 21.0.12.1
Branch HEAD
Compiled by user runner on 2025-09-17T20:37:30Z
Revision ed00d046951a7ecda6429accd3b9c5b2

Upload the original MATLAB dataset

In [16]:

from google.colab import files
import os

if not os.path.exists("/content/exPCA.mat"):
    print("Please upload exPCA.mat")
    files.upload()

print("MATLAB file available:",
      os.path.exists("/content/exPCA.mat"))

print("Scala source available:",
      os.path.exists("/content/Homework6_PCA_Direct.scala"))


Please upload exPCA.mat


Saving jmatio.jar to jmatio.jar
Saving exPCA.mat to exPCA.mat
MATLAB file available: True
Scala source available: True


Run the direct Scala PCA implementation

In [17]:

import os
import subprocess

spark_shell = os.path.join(
    os.environ["SPARK_HOME"],
    "bin",
    "spark-shell"
)

scala_file = "/content/Homework6_PCA_Direct.scala"

result = subprocess.run(
    [spark_shell, "-i", scala_file],
    input=":quit\n",
    text=True,
    capture_output=True,
    timeout=180
)

print("===== SCALA PCA RESULTS =====")
print(result.stdout[-12000:])

print("===== ERRORS =====")
print(result.stderr[-2000:])

print("Exit code:", result.returncode)


===== SCALA PCA RESULTS =====
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Spark context Web UI available at http://0dbabae62689:4040
Spark context available as 'sc' (master = local[*], app id = local-1793203713285).
Spark session available as 'spark'.
java.lang.IllegalArgumentException: newPosition > limit: (1000 > 995)
  at java.base/java.nio.Buffer.createPositionException(Buffer.ja

In [18]:

from google.colab import files
import os

if not os.path.exists("/content/exPCA.mat"):
    print("Please upload exPCA.mat")
    files.upload()

print("MATLAB file available:",
      os.path.exists("/content/exPCA.mat"))

print("Scala source available:",
      os.path.exists("/content/Homework6_PCA_Direct.scala"))


MATLAB file available: True
Scala source available: True


In [19]:

import os
import subprocess

spark_shell = os.path.join(
    os.environ["SPARK_HOME"], "bin", "spark-shell"
)

scala_file = "/content/Homework6_PCA_Direct.scala"

result = subprocess.run(
    [spark_shell, "-i", scala_file],
    input=":quit\n",
    text=True,
    capture_output=True,
    timeout=180
)

print("===== SCALA PCA RESULTS =====")
print(result.stdout[-12000:])

print("===== ERRORS =====")
print(result.stderr[-3000:])

print("Exit code:", result.returncode)


===== SCALA PCA RESULTS =====
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Spark context Web UI available at http://0dbabae62689:4040
Spark context available as 'sc' (master = local[*], app id = local-1793203967409).
Spark session available as 'spark'.
java.lang.IllegalArgumentException: newPosition > limit: (1000 > 995)
  at java.base/java.nio.Buffer.createPositionException(Buffer.ja

Correct the Scala source code

In [20]:

from pathlib import Path

path = Path("/content/Homework6_PCA_Direct.scala")
code = path.read_text()

old = """    b.position(b.position() + padding)
    (tag, bytes)"""

new = """    val remainingPadding =
      math.min(padding, b.remaining())
    b.position(b.position() + remainingPadding)
    (tag, bytes)"""

if old not in code:
    print("Original padding code not found. Check source.")
else:
    code = code.replace(old, new)
    path.write_text(code)
    print("Scala padding fix applied successfully!")


Scala padding fix applied successfully!


Run the corrected Scala program

In [21]:

import os
import subprocess

spark_shell = os.path.join(
    os.environ["SPARK_HOME"],
    "bin",
    "spark-shell"
)

result = subprocess.run(
    [
        spark_shell,
        "-i", "/content/Homework6_PCA_Direct.scala"
    ],
    input=":quit\n",
    text=True,
    capture_output=True,
    timeout=180
)

print("===== SCALA PCA RESULTS =====")
print(result.stdout[-12000:])

print("===== ERRORS =====")
print(result.stderr[-3000:])

print("Exit code:", result.returncode)


===== SCALA PCA RESULTS =====
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Spark context Web UI available at http://0dbabae62689:4040
Spark context available as 'sc' (master = local[*], app id = local-1793204093593).
Spark session available as 'spark'.
===== MATLAB DATA LOADED IN SCALA =====
Rows: 50
Columns: 2
3.38156266663556, 3.389112684892071
4.5278753804032075, 5.854178101169413
